In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

# Set style for matplotlib/seaborn
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Load the cleaned data
df = pd.read_csv('../cleaned_data/master_dataset.csv')

print("Dataset Shape:", df.shape)
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- Summary Statistics for Numerical Features ---")
print(df[['Price', 'Rating']].describe())
print("\n--- Value Counts for Categorical Features ---")
print(df['Category'].value_counts().head(10))

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Histogram and Boxplot for Price
fig, (ax1, ax2) = plt.subplots(1, 2)
sns.histplot(df['Price'], bins=30, kde=True, ax=ax1)
ax1.set_title('Distribution of Book Prices')
sns.boxplot(y=df['Price'], ax=ax2)
ax2.set_title('Boxplot of Book Prices')
plt.tight_layout()
plt.show()

# Identify price outliers using IQR
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.25)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['Price'] < lower_bound) | (df['Price'] > upper_bound)]
print(f"Number of price outliers: {len(outliers)}")

In [ ]:
# Group by Category and analyze
category_stats = df.groupby('Category')['Price'].agg(['mean', 'median', 'count', 'std']).round(2)
category_stats = category_stats.sort_values('mean', ascending=False)
print("Top 5 Most Expensive Categories on Average:")
print(category_stats.head())

# Visualize: Average Price by Top 10 Categories
top_categories = category_stats.head(10)
plt.figure(figsize=(14, 6))
sns.barplot(x=top_categories.index, y=top_categories['mean'])
plt.title('Average Price by Book Category (Top 10)')
plt.xticks(rotation=45)
plt.ylabel('Average Price (£)')
plt.show()

In [ ]:
# Scatter plot: Rating vs. Price
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df['Rating'], y=df['Price'], hue=df['Category'], palette='viridis', alpha=0.6)
plt.title('Book Price vs. Rating')
plt.show()

# Calculate correlation coefficient
correlation = df['Price'].corr(df['Rating'])
print(f"Correlation between Price and Rating: {correlation:.2f}")

In [ ]:
# Example: Are books in the 'Fiction' category priced differently than 'Non-Fiction'?
# First, create the two sample groups
fiction_prices = df[df['Category'] == 'Fiction']['Price'].dropna()
nonfiction_prices = df[df['Category'] == 'Non-Fiction']['Price'].dropna()

# Perform an independent t-test
t_stat, p_value = stats.ttest_ind(fiction_prices, nonfiction_prices, equal_var=False) # Use Welch's t-test

print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis: There is a significant difference in price.")
else:
    print("Fail to reject the null hypothesis: No significant difference in price.")

In [ ]:
# Interactive Box Plot
fig = px.box(df, x='Category', y='Price', title='Book Prices by Category (Interactive)',
             hover_data=['Title'], color='Category')
fig.update_xaxes(tickangle=45)
fig.show()

# Interactive Scatter Plot
fig = px.scatter(df, x='Rating', y='Price', color='Category',
                 title='Rating vs. Price by Category', trendline="ols", # Adds regression line
                 hover_name='Title')
fig.show()

In [ ]:
fig.write_html("../visualizations/interactive_price_boxplot.html")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Prepare data: Can we predict price from rating and other features?
# Handle missing values and select features
model_df = df[['Price', 'Rating', 'Page_Count']].dropna() # Assuming you scraped page count
X = model_df[['Rating', 'Page_Count']] # Independent variables
y = model_df['Price']                   # Dependent variable (target)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions and evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R-squared Score: {r2:.3f}")
print(f"Root Mean Squared Error: £{rmse:.2f}")
print(f"Model Coefficients: {dict(zip(X.columns, model.coef_))}")